# 1.1

In [1]:
import pymongo
from pymongo import MongoClient

client = MongoClient('host.docker.internal', 27017)
db = client['fit3182_db']  




In [2]:
import pandas as pd

In [3]:
vehicle_df = pd.read_csv("/home/student/work/ass2/data/vehicle.csv")
print(vehicle_df.head())

  car_plate           owner_name                              owner_addr  \
0     FT 02          Goh Mei Wei     943 Jalan Bukit Mawar, Kuala Lumpur   
1   DQZ 793  Muhammad bin Liyana  946 Jalan Bukit Jelutong, Kuala Lumpur   
2   GUT 393      Haziq bin Azlan   900 Jalan Bukit Bintang, Kuala Lumpur   
3     KN 37   Haziq binti Liyana     285 Jalan Bukit Mawar, Kuala Lumpur   
4      CJ 6         Wong Mei Hui  549 Jalan Bukit Jelutong, Kuala Lumpur   

  vechicle_type    registration_date  
0         Coupe  2006-08-22T03:18:00  
1           SUV  2008-04-13T09:01:23  
2         Sedan  2008-01-20T20:50:45  
3         Coupe  1999-10-15T22:28:10  
4         Coupe  2007-01-25T22:36:25  


In [4]:
vehicles = vehicle_df.to_dict(orient="records")
db.vehicle.insert_many(vehicles)


In [5]:
camera_df = pd.read_csv("/home/student/work/ass2/data/camera.csv")
print(camera_df.head())

   camera_id  latitude   longitude  position  speed_limit
0          1  2.157731  102.660100     152.5          110
1          2  2.162419  102.652455     153.5          110
2          3  2.167353  102.644914     154.5           90


In [6]:
cameras = camera_df.to_dict(orient="records")
db.cameras.insert_many(cameras)

In [7]:
#violation_df = pd.read_csv("/home/student/work/ass2/data/camera_event_historic.csv")
#print(violation_df.head())

In [8]:
# violations = violation_df.to_dict(orient="records")
# db.violations.insert_many(violations)

In [9]:
from pymongo import MongoClient

client = MongoClient("host.docker.internal", 27017)  # ← needs to be assigned to `client`
db = client['fit3182_db']



print("Collections:", db.list_collection_names())


# Print 5 documents from each collection
print(" vehicle collection:")
for doc in db.vehicle.find().limit(5):
    print(doc)

print("\n cameras collection:")
for doc in db.cameras.find().limit(5):
    print(doc)

print("\n violations collection:")
for doc in db.violations.find().limit(5):
    print(doc)

Collections: ['vehicle', 'cameras']
 vehicle collection:
{'_id': ObjectId('6829795bbd84b3ab2e8d38c1'), 'car_plate': 'FT 02', 'owner_name': 'Goh Mei Wei', 'owner_addr': '943 Jalan Bukit Mawar, Kuala Lumpur', 'vechicle_type': 'Coupe', 'registration_date': '2006-08-22T03:18:00'}
{'_id': ObjectId('6829795bbd84b3ab2e8d38c2'), 'car_plate': 'DQZ 793', 'owner_name': 'Muhammad bin Liyana', 'owner_addr': '946 Jalan Bukit Jelutong, Kuala Lumpur', 'vechicle_type': 'SUV', 'registration_date': '2008-04-13T09:01:23'}
{'_id': ObjectId('6829795bbd84b3ab2e8d38c3'), 'car_plate': 'GUT 393', 'owner_name': 'Haziq bin Azlan', 'owner_addr': '900 Jalan Bukit Bintang, Kuala Lumpur', 'vechicle_type': 'Sedan', 'registration_date': '2008-01-20T20:50:45'}
{'_id': ObjectId('6829795bbd84b3ab2e8d38c4'), 'car_plate': 'KN 37', 'owner_name': 'Haziq binti Liyana', 'owner_addr': '285 Jalan Bukit Mawar, Kuala Lumpur', 'vechicle_type': 'Coupe', 'registration_date': '1999-10-15T22:28:10'}
{'_id': ObjectId('6829795bbd84b3ab2e8

#### Vehicle Collection Design

**Purpose**:  
Stores static metadata about registered vehicles such as ownership, address, type, and registration date. This collection is primarily used to enrich event or violation data.

**Schema**:
```json
{
  "car_plate": "FT 02",
  "owner_name": "Goh Mei Wei",
  "owner_addr": "943 Jalan Bukit Mawar, Kuala Lumpur",
  "vehicle_type": "Coupe",
  "registration_date": "2006-08-22T03:18:00"
}

**Indexes:**:  
Field: car_plate (Ascending)
Type: Unique index
Purpose: Efficient lookups for joining vehicle data with events or violations

**Indexes:**: 
Key: car_plate
Type: Hashed
Rationale: Provides good distribution due to uniqueness and randomness of plate numbers.

Retention Policy:
Data is static and retained indefinitely.

#### Camera Collection Design 

**Purpose**:  
Stores static configuration and geolocation data for each camera, including speed limits. Used to interpret event and violation context.

**Schema**:
```json

{
  "camera_id": 2,
  "latitude": 2.162418757,
  "longitude": 102.6524549,
  "position": 153.5,
  "speed_limit": 110.0
}

**Index**:
Fields: camera_id (ascending)
Type: Unique index
Purpose: Quickly locate camera configuration by ID

**Shard Key**:
Chosen key: camera_id
Type: Ranged (or Hashed if IDs are uniformly accessed)
Rationale: Ranged keys enable efficient range queries over camera segments

**Retention Policy**:
Static data — retained indefinitely.

#### Camera Collection Design Collection Design

Purpose:
Stores violations generated when a vehicle exceeds the average legal speed between two cameras. Supports enforcement and analytics.

**Schema**:
```json

{
  "violation_id": "v001",
  "car_plate": "FT 02",
  "camera_id_start": 1,
  "camera_id_end": 3,
  "timestamp_start": "2024-01-01T08:00:00",
  "timestamp_end": "2024-01-01T08:03:12",
  "average_speed": 125.3
}

**Index**:
Fields: car_plate, timestamp_start (compound index)
Type: Compound index
Purpose: Support querying violations by car and time range
Shard Key:

**Shard Key**:
Chosen key: violation_id
Type: Hashed
Rationale: Violations are independently identified; hashed key ensures balance

**Retention Policy**:
Retain for 5–7 years per enforcement policy. Can be archived afterward.

# 1.2

1. vehicle ↔ violations

   Relationship: 1-to-many

   Join field: car_plate

   Design: No embedding, Referencing by car_plate


2. camera ↔ violations

   Relationship: 1-to-many (each violation involves 2 cameras)

   Join fields: camera_id_start, camera_id_end

   Design:  No embedding, Referencing by camera_id
   

3. camera_event_historic (if reused)

   Relationship: Intermediate source used to build violations

   No long-term relation once violations are created

# 1.3

### Consistency and Idempotency

Idempotent Writes:
Yes, the model supports idempotent writes, particularly in the violations collection.

We use an upsert pattern in the streaming application:


db.violations.update_one(
    {"car_plate": row["car_plate"], "date": row["timestamp_start"].date()},
    {"$set": row},
    upsert=True
)

This ensures:

No duplicate violation records for the same car on the same day.

If a newer violation for the same car on the same date arrives (e.g., from a different camera pair), the record is updated rather than duplicated.

Consistency Guarantees:
Because vehicle and camera metadata are static and read-only, referencing them ensures strong consistency without synchronization overhead.
Streaming writes to violations are handled in append/upsert mode, so eventual consistency is acceptable given the real-time nature of the system.

### Scalability and Fault-Tolerance
1. 
High Ingest Rates:
Yes — the use of Kafka and Spark Structured Streaming enables scalable real-time ingestion.
MongoDB's document model and shardable design (e.g., hashed car_plate or violation_id) supports horizontal scaling.

2. 
Low-Latency Lookups:

vehicle: Indexed by car_plate → fast joins with camera events

camera: Indexed by camera_id → fast lookup of position/speed

violations: Compound index on car_plate and timestamp_start → efficient time-range queries

3. 
Fault Tolerance:

Kafka provides message durability and replayability

Spark uses checkpointing and exactly-once semantics

MongoDB upsert logic ensures resilience against message reprocessing or duplicates

### Trade-Offs and Justifications

1. 
Referencing instead of Embedding:
Referencing static collections (vehicle, camera) avoids data duplication and supports schema flexibility. Although this incurs a slight join cost during stream processing, Spark handles this efficiently.

2. 
Upsert over Insert-only Writes:
Using upserts allows us to handle duplicate or late-arriving messages gracefully and ensures idempotent writes — at the cost of slightly more complex logic.

3. 
Hashed Shard Keys:
Although range-based sharding might allow ordered queries, we use hashed keys (e.g., on car_plate) to ensure balanced distribution across shards, which benefits high write volume scenarios.

4. 
Preloading Static Data into MongoDB:
Camera and vehicle metadata are preloaded and not included in each violation record. This reduces message size and avoids redundancy, at the cost of requiring a lookup during streaming.

5. 
No Embedding of Owner Info in Violations:
Avoids syncing issues if owner details change and keeps violation records small and focused. This introduces a minor join cost during visualization, which is acceptable.

#### Streaming Section

#### Step 1

In [10]:
import os

# Include both Kafka and Mongo connectors BEFORE Spark starts
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages '
    'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,'
    'org.mongodb.spark:mongo-spark-connector_2.12:3.0.1 '
    'pyspark-shell'
)

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AWAS Streaming App") \
    .master("local[*]") \
    .getOrCreate()

In [11]:
from pyspark.sql.functions import col, from_json, expr
from pyspark.sql.types import StructType, StringType, IntegerType, FloatType, TimestampType

schema = StructType() \
    .add("event_id", StringType()) \
    .add("batch_id", IntegerType()) \
    .add("car_plate", StringType()) \
    .add("camera_id", IntegerType()) \
    .add("timestamp", StringType()) \
    .add("speed_reading", FloatType()) \
    .add("producer", StringType())

def read_topic(topic_name):
    return spark.readStream.format("kafka") \
        .option("kafka.bootstrap.servers", "host.docker.internal:9092") \
        .option("subscribe", topic_name) \
        .load() \
        .selectExpr("CAST(value AS STRING) as json_str") \
        .select(from_json(col("json_str"), schema).alias("data")) \
        .select("data.*")

stream_a = read_topic("camera_event_a")
stream_b = read_topic("camera_event_b")
stream_c = read_topic("camera_event_c")

all_streams = stream_a.union(stream_b).union(stream_c) \
    .withColumn("timestamp", expr("to_timestamp(timestamp)"))


*check if correct*

In [12]:
all_streams.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- batch_id: integer (nullable = true)
 |-- car_plate: string (nullable = true)
 |-- camera_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- speed_reading: float (nullable = true)
 |-- producer: string (nullable = true)



In [13]:
camera_df = spark.read.format("mongo") \
    .option("uri", "mongodb://host.docker.internal:27017/fit3182_db.cameras") \
    .load()


In [14]:
events_with_meta = all_streams.join(camera_df, on="camera_id", how="left")

In [15]:
keyed = events_with_meta \
    .withWatermark("timestamp", "5 minutes") \
    .select("car_plate", "camera_id", "timestamp", "position", "speed_reading", "speed_limit") \
    .groupBy("car_plate") 

In [16]:
from pyspark.sql.functions import col
from datetime import datetime
import pymongo

def detect_violation_batch(df, epoch_id):
    if df.isEmpty():
        return

    pdf = df.toPandas()
    print(f"[Epoch {epoch_id}] Processing {len(pdf)} sessions")
    
    violations = []

    for _, row in pdf.iterrows():
        car_plate = row["car_plate"]
        date_key = row["session_start"].date().isoformat()
        events = sorted(row["events"], key=lambda e: e["timestamp"])

        for i in range(len(events) - 1):
            start = events[i]
            end = events[i + 1]

            if start["camera_id"] == end["camera_id"] or start["timestamp"] >= end["timestamp"]:
                continue

            dist_km = abs(end["position"] - start["position"])
            time_hr = (end["timestamp"] - start["timestamp"]).total_seconds() / 3600.0
            avg_speed = dist_km / time_hr if time_hr > 0 else 0

            # Check violation rule
            if start["speed_reading"] > start["speed_limit"] or avg_speed > end["speed_limit"]:
                violations.append({
                    "car_plate": car_plate,
                    "date": date_key,
                    "camera_id_start": start["camera_id"],
                    "camera_id_end": end["camera_id"],
                    "timestamp_start": start["timestamp"],
                    "timestamp_end": end["timestamp"],
                    "max_speed": max(start["speed_reading"], avg_speed)
                })
                break  # stop after first violation per car per day

    # Write to MongoDB
    if violations:
        from pymongo import MongoClient
        client = MongoClient("mongodb://localhost:27017")
        db = client["fit3182_db"]
        for violation in violations:
            db["violations"].update_one(
                {"car_plate": violation["car_plate"], "date": violation["date"]},
                {"$set": violation},
                upsert=True
            )
        client.close()



In [17]:
from pymongo import MongoClient
from datetime import datetime

class MongoWriter:
    def open(self, part_id, epoch_id):
        self.client = MongoClient("mongodb://localhost:27017")
        self.db = self.client["awas_db"]
        return True

    def process(self, row):
        date_key = row.timestamp_start.date().isoformat()
        self.db.violations.update_one(
            {"car_plate": row.car_plate, "date": date_key},
            {"$set": row.asDict()},
            upsert=True
        )

    def close(self, err):
        self.client.close()


In [18]:
from pyspark.sql.functions import session_window, expr

sessioned = events_with_meta \
    .withWatermark("timestamp", "5 minutes") \
    .groupBy(session_window("timestamp", "10 minutes"), "car_plate") \
    .agg(
        expr("min(timestamp) as session_start"),
        expr("max(timestamp) as session_end"),
        expr("max(speed_reading) as max_speed"),
        expr("collect_list(struct(camera_id, timestamp, position, speed_limit, speed_reading)) as events")
    )

# Production: Write sessions to violation detection logic
sessioned.writeStream \
    .foreachBatch(detect_violation_batch) \
    .outputMode("complete") \
    .start()

# Development/testing: View the session aggregation output
sessioned.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()





In [19]:
import time
from pymongo import MongoClient

client = MongoClient("mongodb://host.docker.internal:27017")
db = client["fit3182_db"]
seen_ids = set()

while True:
    violations = db["violations"].find().sort("timestamp_end", -1).limit(5)
    for v in violations:
        if v["_id"] not in seen_ids:
            print("[NEW VIOLATION]", v)
            seen_ids.add(v["_id"])
    time.sleep(3)


/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)
/opt/conda/lib/python3.8/site-packages/pyspark/sql/pandas/conversion.py:248: FutureWarning: Passing unit-less datetime64 dtype to .astype is deprecated and will raise in a future version. Pass 'datetime64[ns]' instead
  series = series.astype(t, copy=False)


[Epoch 1] Processing 71 sessions


KeyboardInterrupt: 